# Stabilized joint training — 500-step smoke only
เปิดแท็บไว้และอย่าให้เครื่อง sleep. อัปโหลด bundle เดิมไป My Drive/thai_dsr_colab แล้วเลือก GPU runtime.

รัน setup cells 1–5 ตามลำดับ จากนั้นรัน smoke 500 steps: warmup 100, ramp 200, full GAN weight 200. Clip norm 1.0, mel 45, content 0.5, GAN 1, FM 2. Validate 3 VAL utterances ทุก 100 steps และ step 0. เทียบกับ baseline ของ VAL ชุดเดียวกัน; ห้ามใช้ TEST STOI 0.180 เป็น baseline ของ VAL.

โน้ตบุ๊กแนบ source ที่แก้แล้วในตัว เขียนเป็น module ชื่อใหม่ตาม SHA-256 ไม่ทับ training code เดิม และสร้าง checkpoint subfolder ใหม่เสมอ ไม่ resume จากรอบที่เสื่อมคุณภาพ และไม่มีเซลล์เริ่ม long run อัตโนมัติ.

เซลล์สุดท้ายประเมิน 3 variants บน 8 TEST utterances เดิม ด้วย inference/metrics จริง ใช้ checkpoint step 500 ที่เพิ่งสร้างโดยตรง. ผล CPU ไม่รับประกันจะตรง CUDA แบบ bitwise.

In [ ]:
# 1. GPU
import torch
print('torch:', torch.__version__)
print('torch.cuda.is_available():', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)


In [ ]:
# 2. Google Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/thai_dsr_colab')  # EDIT if needed
BUNDLE = DRIVE_ROOT / 'colab_transfer_bundle.zip'
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
assert BUNDLE.is_file(), f'Upload the zip to {BUNDLE}'


In [ ]:
# 3. Clone project (safe to rerun in the same runtime)
import os, subprocess, sys
REPO = Path('/content/thai-dsr')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/KankaveeRamsri/thai-dsr.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


In [ ]:
# 4. Dependencies: preserve Colab's matching CUDA torch/torchaudio builds.
# Use subprocesses so package changes do not leave stale imports in the trainer.
import importlib.metadata as metadata
constraints = Path('/content/thai_dsr_constraints.txt')
constraints.write_text(''.join(f'{name}=={metadata.version(name)}\n' for name in ('torch', 'torchaudio')))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt',
                'PyYAML>=6.0', 'transformers==5.13.1', '-c', str(constraints)], check=True)
subprocess.run([sys.executable, '-c',
    'import torch, torchaudio, yaml; from transformers import Wav2Vec2Model; '
    'assert torch.cuda.is_available(); print(torch.__version__, torchaudio.__version__)'], check=True)


In [ ]:
# 5. Extract at repo root and verify every bundled file (SHA-256).
import hashlib, json, shutil, zipfile
local_zip = Path('/content/colab_transfer_bundle.zip')
shutil.copyfile(BUNDLE, local_zip)
with zipfile.ZipFile(local_zip) as archive:
    for entry in archive.infolist():
        target = (REPO / entry.filename).resolve()
        assert target.is_relative_to(REPO.resolve()), entry.filename
    archive.extractall(REPO)
local_zip.unlink()
manifest = json.loads((REPO / 'scripts/colab_bundle_manifest.json').read_text())
for entry in manifest['files']:
    p = REPO / entry['path']
    assert p.stat().st_size == entry['bytes'], p
    digest = hashlib.sha256()
    with p.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b''):
            digest.update(block)
    assert digest.hexdigest() == entry['sha256'], f'Checksum mismatch: {p}'
print('Verified', len(manifest['files']), 'files; HiFi-GAN revision:', manifest['hifigan_revision'])
# The trainer confines outputs to this tree. A symlink sends writes directly to Drive.
output_link = REPO / 'results/checkpoints/joint_finetune'
if output_link.is_symlink():
    assert output_link.resolve() == CHECKPOINT_ROOT.resolve()
elif output_link.exists():
    raise RuntimeError(f'{output_link} already exists; preserve its contents before linking Drive.')
else:
    output_link.symlink_to(CHECKPOINT_ROOT, target_is_directory=True)
assert output_link.resolve() == CHECKPOINT_ROOT.resolve()
# Download the public frozen encoder automatically (about 1.2 GiB weights).
# The trainer reads this cache by model name; no local Mac cache is required.
os.environ['HF_HOME'] = '/content/thai_dsr_hf'
os.environ.pop('HF_HUB_OFFLINE', None)
subprocess.run([sys.executable, '-c',
    'from huggingface_hub import snapshot_download; '
    'snapshot_download("airesearch/wav2vec2-large-xlsr-53-th", '
    'allow_patterns=["config.json", "preprocessor_config.json", "model.safetensors", "pytorch_model.bin"])'], check=True)
subprocess.run([sys.executable, '-m', 'src.training.joint_finetune', '--help'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_joint_finetune', '-v'], check=True)


In [ ]:
# Save the exact revised trainer under a NEW, content-addressed module name.
TRAINER_SOURCE = '"""Joint mapper/HiFi-GAN fine-tuning. Existing models and datasets remain untouched."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nimport gc\nfrom datetime import datetime\nimport hashlib\nimport itertools\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nimport torch.nn.functional as F\nimport torchaudio.functional as AF\n\nfrom src.inference.run import load_mapper, MEL_CLAMP_MIN, MEL_CLAMP_MAX\nfrom src.models.encoder import Wav2Vec2ContentEncoder\nfrom src.training.dataset import interpolate_embedding\nfrom src.training.finetune_hifigan import (\n    Generator, MultiPeriodDiscriminator, MultiScaleDiscriminator,\n    discriminator_loss, feature_loss, generator_loss, load_config,\n    choose_device, set_requires_grad,\n)\nfrom src.utils.mel import compute_mel_tensor\nfrom src.evaluation.metrics import compute_stoi, compute_pesq\n\nROOT = Path(__file__).resolve().parents[2]\n\n\ndef path(value):\n    p = Path(value)\n    return p if p.is_absolute() else ROOT / p\n\n\ndef sha256(p):\n    h = hashlib.sha256()\n    with open(p, \'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef default_mapper():\n    # The latest controlled pipeline experiment is authoritative, not the older CLI default.\n    p = ROOT / \'results/audio_compare/w6_vocoder_ft/provenance.json\'\n    if not p.is_file():\n        raise FileNotFoundError(\'Pass --mapper-checkpoint; pipeline provenance is absent\')\n    metadata = json.loads(p.read_text())\n    checkpoint = path(metadata[\'mapper\'])\n    if sha256(checkpoint) != metadata[\'mapper_sha256\']:\n        raise ValueError(\'Pipeline mapper differs from recorded provenance; pass --mapper-checkpoint\')\n    return checkpoint\n\n\ndef load_pairs(manifest, split_file):\n    splits = json.loads(path(split_file).read_text())[\'splits\']\n    ids = list(itertools.chain.from_iterable(splits.values()))\n    if len(ids) != len(set(ids)):\n        raise ValueError(\'Split leakage or duplicate IDs\')\n    with path(manifest).open() as f:\n        rows = [r for r in csv.DictReader(f) if r[\'severity\'] == \'severe\']\n    by_id = {r[\'utterance_id\']: r for r in rows}\n    if len(rows) != len(by_id) or set(by_id) != set(ids):\n        raise ValueError(\'Manifest and split IDs must match exactly, one severe pair per ID\')\n    for r in rows:\n        for key in (\'clean_path\', \'distorted_path\'):\n            if not path(r[key]).is_file():\n                raise FileNotFoundError(r[key])\n    return {s: [by_id[i] for i in sorted(splits[s])] for s in (\'train\', \'val\', \'test\')}\n\n\ndef load_wave(p, sr=22050):\n    x, original_sr = sf.read(path(p), dtype=\'float32\', always_2d=True)\n    x = torch.from_numpy(x.mean(axis=1)).unsqueeze(0)\n    if original_sr != sr:\n        x = AF.resample(x, original_sr, sr)\n    return x\n\n\nclass FrozenContent:\n    """Same frozen XLSR for input and content; generated branch retains autograd."""\n    def __init__(self, device):\n        self.device = device\n        self.wrapper = Wav2Vec2ContentEncoder(device=device, layer=9)\n        self.model = self.wrapper.model.eval()\n        self.model.requires_grad_(False)\n        # hidden_states[9] is captured BEFORE block 10; keep ten blocks so the\n        # stable-layer-norm encoder\'s final normalization cannot change layer 9.\n        probe = torch.linspace(-0.1, 0.1, 8000, device=device).unsqueeze(0)\n        with torch.no_grad():\n            expected = self.model(probe, output_hidden_states=True).hidden_states[9]\n            self.model.encoder.layers = torch.nn.ModuleList(list(self.model.encoder.layers[:10]))\n            actual = self.model(probe, output_hidden_states=True).hidden_states[9]\n            torch.testing.assert_close(actual, expected, rtol=0, atol=0)\n        print(\'Verified exact layer-9 equivalence after omitting unused upper blocks.\', flush=True)\n\n    def encode_wave(self, audio, sr=22050):\n        # CPU resampling is differentiable and avoids backend-specific sinc limitations.\n        x = AF.resample(audio.cpu(), sr, 16000).to(self.device) if sr != 16000 else audio.to(self.device)\n        x = (x - x.mean(-1, keepdim=True)) / torch.sqrt(x.var(-1, unbiased=False, keepdim=True) + 1e-7)\n        return self.model(x, output_hidden_states=True).hidden_states[9]\n\n    @torch.no_grad()\n    def input_embedding(self, p):\n        # Reuse exactly the existing feature extractor and input resampling path.\n        return self.wrapper.encode(str(path(p)))\n\n\ndef loss_mel(wave):\n    # Explicit CPU STFT keeps the complex transform portable; .cpu() preserves gradients.\n    return compute_mel_tensor(wave.squeeze(1).cpu(), fmax=None)\n\n\ndef grad_norm(module):\n    norms = [p.grad.detach().float().norm().cpu() for p in module.parameters() if p.grad is not None]\n    value = torch.stack(norms).norm() if norms else torch.tensor(0.)\n    if not torch.isfinite(value) or value <= 0:\n        raise RuntimeError(f\'Invalid/zero gradient in {type(module).__name__}: {value}\')\n    return float(value)\n\n\ndef gan_scale(step, warmup_steps, ramp_steps):\n    """Global, 1-based update schedule; resume continues at its saved step."""\n    if min(warmup_steps, ramp_steps) < 0:\n        raise ValueError(\'GAN warmup/ramp steps must be nonnegative\')\n    if step <= warmup_steps:\n        return 0.0\n    return min(1.0, (step - warmup_steps) / ramp_steps) if ramp_steps else 1.0\n\n\ndef clip_optimizer_gradients(modules, max_norm):\n    """Clip the combined optimizer norm, and measure it again after clipping."""\n    if not np.isfinite(max_norm) or max_norm <= 0:\n        raise ValueError(\'grad-clip must be finite and positive\')\n    parameters = [p for module in modules for p in module.parameters() if p.grad is not None]\n    if not parameters:\n        raise RuntimeError(\'Optimizer has no gradients to clip\')\n    before = float(torch.nn.utils.clip_grad_norm_(parameters, max_norm, error_if_nonfinite=True))\n    after = float(torch.stack([p.grad.detach().float().norm() for p in parameters]).norm())\n    if not np.isfinite(after) or after > max_norm * 1.001:\n        raise RuntimeError(f\'Gradient clipping failed: {after} > {max_norm}\')\n    return before, after\n\n\ndef weighted_loss_gradients(terms, predicted):\n    """Compare loss gradients at the SAME mapper output, before optimizer clipping."""\n    result = {}\n    for name, term in terms.items():\n        if not term.requires_grad:  # GAN branches are not evaluated during warmup.\n            result[name] = 0.0\n            continue\n        gradient = torch.autograd.grad(term, predicted, retain_graph=True)[0]\n        norm = float(gradient.detach().float().norm())\n        if not np.isfinite(norm):\n            raise RuntimeError(f\'Non-finite {name} gradient at mapper output\')\n        result[name] = norm\n    return result\n\n\ndef synchronize(device):\n    if device.type == \'mps\':\n        torch.mps.synchronize()\n    elif device.type == \'cuda\':\n        torch.cuda.synchronize()\n\n\ndef save_joint(out, step, mapper, generator, mpd, msd, og, od, args, config, init, rng):\n    destination = out / f\'joint_{step:08d}.pt\'\n    if destination.exists():\n        raise FileExistsError(destination)\n    if shutil.disk_usage(out).free < 2 * 1024**3:\n        raise OSError(\'Less than 2 GiB free; refusing checkpoint write\')\n    payload = dict(step=step, mapper=mapper.state_dict(), generator=generator.state_dict(),\n                   mpd=mpd.state_dict(), msd=msd.state_dict(), optim_g=og.state_dict(), optim_d=od.state_dict(),\n                   args=vars(args), vocoder_config=dict(config), initialization=init,\n                   torch_rng=torch.get_rng_state(), sampling_rng=rng.getstate())\n    temporary = destination.with_suffix(\'.tmp\')\n    torch.save(payload, temporary)\n    os.replace(temporary, destination)\n    print(f\'Saved joint checkpoint: {destination}\', flush=True)\n\n\n@torch.no_grad()\ndef validate(step, rows, get_item, mapper, generator, device, out, baseline=None):\n    mapper.eval()\n    generator.eval()\n    scores = []\n    for row in rows:\n        embedding, clean, distorted_frames = get_item(row)\n        inputs = interpolate_embedding(embedding, distorted_frames).unsqueeze(0).to(device)\n        mel = mapper(inputs).clamp(MEL_CLAMP_MIN, MEL_CLAMP_MAX).transpose(1, 2)\n        audio = generator(mel).squeeze().cpu().numpy()\n        audio = np.clip(audio, -1, 1)\n        folder = out / \'validation\' / f\'step_{step:08d}\'\n        folder.mkdir(parents=True, exist_ok=True)\n        sf.write(folder / f\'{row["utterance_id"]}.wav\', audio, 22050, subtype=\'PCM_16\')\n        # Measure saved PCM audio, matching the previous evaluation protocol.\n        saved, _ = sf.read(folder / f\'{row["utterance_id"]}.wav\', dtype=\'float32\')\n        reference = AF.resample(clean, 22050, 16000).squeeze().numpy()\n        estimate = AF.resample(torch.from_numpy(saved), 22050, 16000).numpy()\n        scores.append(dict(utterance_id=row[\'utterance_id\'], stoi=float(compute_stoi(reference, estimate)),\n                           pesq=float(compute_pesq(reference, estimate))))\n    record = dict(step=step, items=scores, stoi=float(np.mean([s[\'stoi\'] for s in scores])),\n                  pesq=float(np.mean([s[\'pesq\'] for s in scores])))\n    if not all(np.isfinite(s[k]) for s in scores for k in (\'stoi\', \'pesq\')):\n        raise RuntimeError(\'Non-finite validation metric\')\n    if baseline is not None:\n        record[\'baseline_step\'] = baseline[\'step\']\n        record[\'delta_stoi\'] = record[\'stoi\'] - baseline[\'stoi\']\n        record[\'delta_pesq\'] = record[\'pesq\'] - baseline[\'pesq\']\n    with (out / \'validation.jsonl\').open(\'a\') as f:\n        f.write(json.dumps(record) + \'\\n\')\n    print(\'VALIDATION \' + json.dumps(record), flush=True)\n    mapper.train()\n    generator.train()\n    return record\n\n\ndef train(args):\n    os.chdir(ROOT)\n    device = choose_device(args.device)\n    torch.set_num_threads(4)\n    torch.manual_seed(args.seed)\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    rng = random.Random(args.seed)\n    if args.segment_samples < 8192 or args.segment_samples % 256:\n        raise ValueError(\'segment-samples must be >=8192 and divisible by 256\')\n    if min(args.adv_weight, args.fm_weight, args.mel_weight, args.content_weight) <= 0:\n        raise ValueError(\'All four loss weights must be positive\')\n    if args.max_steps < 1 or args.val_items < 1:\n        raise ValueError(\'Positive max-steps and val-items required\')\n    gan_scale(1, args.gan_warmup_steps, args.gan_ramp_steps)\n    if not np.isfinite(args.grad_clip) or args.grad_clip <= 0:\n        raise ValueError(\'grad-clip must be finite and positive\')\n    if min(args.validation_interval, args.checkpoint_interval, args.loss_grad_interval) < 0 or args.log_interval < 1:\n        raise ValueError(\'Intervals must be nonnegative; log-interval must be positive\')\n    mapper_path = path(args.mapper_checkpoint) if args.mapper_checkpoint else default_mapper()\n    pretrained = path(\'results/checkpoints/hifigan_thai\' if args.vocoder_init == \'thai\' else \'vendor/hifi-gan/checkpoints/UNIVERSAL_V1\')\n    gpath = pretrained / (\'g_00010000\' if args.vocoder_init == \'thai\' else \'g_02500000\')\n    dpath = path(args.discriminator_checkpoint) if args.discriminator_checkpoint else pretrained / \'do_latest\'\n    config = load_config(pretrained)\n    assert (config.sampling_rate, config.hop_size, config.num_mels) == (22050, 256, 80)\n    pairs = load_pairs(args.manifest, args.splits)\n    out = path(args.output_dir) if args.output_dir else ROOT / \'results/checkpoints/joint_finetune\' / datetime.now().strftime(\'run_%Y%m%d_%H%M%S\')\n    allowed = ROOT / \'results/checkpoints/joint_finetune\'\n    if not out.resolve().is_relative_to(allowed.resolve()) or out.resolve() == allowed.resolve():\n        raise ValueError(\'Use a NEW run subdirectory under results/checkpoints/joint_finetune\')\n    out.mkdir(parents=True, exist_ok=False)\n    init = dict(mapper=str(mapper_path), mapper_sha256=sha256(mapper_path), generator=str(gpath),\n                generator_sha256=sha256(gpath), discriminator=str(dpath) if dpath.is_file() else \'random\',\n                split_counts={k: len(v) for k, v in pairs.items()}, device=str(device),\n                training_source_sha256=sha256(Path(__file__)))\n    (out / \'run.json\').write_text(json.dumps(dict(args=vars(args), initialization=init), indent=2))\n    print(json.dumps(init, indent=2), flush=True)\n    content = FrozenContent(torch.device(args.content_device))\n    gc.collect()\n    if device.type == "mps":\n        torch.mps.empty_cache()\n    mapper = load_mapper(mapper_path, device, ROOT/\'configs/model.yaml\', ROOT/\'configs/train.yaml\', layer=9).train()\n    generator = Generator(config).to(device).train()\n    generator.load_state_dict(torch.load(gpath, map_location=\'cpu\', weights_only=True)[\'generator\'])\n    mpd, msd = MultiPeriodDiscriminator().to(device), MultiScaleDiscriminator().to(device)\n    if dpath.is_file():\n        state = torch.load(dpath, map_location=\'cpu\', weights_only=True, mmap=True)\n        if args.vocoder_init == \'thai\' and not args.discriminator_checkpoint and state[\'steps\'] != 10000:\n            raise ValueError(\'do_latest does not match g_00010000\')\n        mpd.load_state_dict(state[\'mpd\'])\n        msd.load_state_dict(state[\'msd\'])\n        del state\n    elif args.vocoder_init == \'thai\' or args.discriminator_checkpoint:\n        raise FileNotFoundError(dpath)\n    else:\n        print(\'UNIVERSAL_V1 has no discriminator checkpoint: initializing MPD/MSD randomly.\', flush=True)\n    og = torch.optim.AdamW([{\'params\': mapper.parameters(), \'lr\': args.mapper_lr},\n                           {\'params\': generator.parameters(), \'lr\': args.generator_lr}],\n                          betas=(config.adam_b1, config.adam_b2), weight_decay=0)\n    od = torch.optim.AdamW(itertools.chain(mpd.parameters(), msd.parameters()), lr=args.discriminator_lr,\n                          betas=(config.adam_b1, config.adam_b2), weight_decay=0)\n    start = 0\n    if args.resume:\n        state = torch.load(path(args.resume), map_location=\'cpu\', weights_only=True)\n        for key, model in [(\'mapper\', mapper), (\'generator\', generator), (\'mpd\', mpd), (\'msd\', msd)]:\n            model.load_state_dict(state[key])\n        og.load_state_dict(state[\'optim_g\'])\n        od.load_state_dict(state[\'optim_d\'])\n        start = int(state[\'step\'])\n        rng.setstate(state[\'sampling_rng\'])\n        torch.set_rng_state(state[\'torch_rng\'])\n        del state\n        if start >= args.max_steps:\n            raise ValueError(\'max-steps must exceed resume step\')\n    cache = {}\n    def get_item(row):\n        uid = row[\'utterance_id\']\n        if uid not in cache:\n            emb = content.input_embedding(row[\'distorted_path\'])\n            clean = load_wave(row[\'clean_path\'])\n            distorted = load_wave(row[\'distorted_path\'])\n            cache[uid] = (emb, clean, distorted.shape[-1] // 256)\n        return cache[uid]\n    val_rows = pairs[\'val\'][:args.val_items]\n    baseline = validate(start, val_rows, get_item, mapper, generator, device, out)\n    validations = [baseline]\n    times = []\n    gradient_checks = {}\n    for step in range(start + 1, args.max_steps + 1):\n        synchronize(device)\n        started = time.perf_counter()\n        row = rng.choice(pairs[\'train\'])\n        emb, clean, _ = get_item(row)\n        frames = clean.shape[-1] // 256\n        segment_frames = min(args.segment_samples // 256, frames)\n        begin = rng.randint(0, frames - segment_frames)\n        # Frozen embeddings retain whole-utterance context. Train the BiLSTM on\n        # a bounded window with context on both sides, avoiding variable-length\n        # MPS backward graphs and memory growth on 8 GiB machines.\n        aligned = interpolate_embedding(emb, frames).T.unsqueeze(0)\n        context_frames = 32\n        aligned = F.pad(aligned, (context_frames, context_frames), mode=\'replicate\')\n        inputs = aligned[:, :, begin:begin+segment_frames+2*context_frames].transpose(1, 2).to(device)\n        predicted = mapper(inputs).clamp(MEL_CLAMP_MIN, MEL_CLAMP_MAX)\n        predicted = predicted[:, context_frames:context_frames+segment_frames].transpose(1, 2)\n        generated = generator(predicted)\n        real = clean[:, begin*256:(begin+segment_frames)*256].unsqueeze(1).to(device)\n        assert generated.shape == real.shape\n        for d in (mpd, msd):\n            set_requires_grad(d, True)\n            d.train()\n        od.zero_grad(set_to_none=True)\n        disc = 0\n        for d in (mpd, msd):\n            dr, df, _, _ = d(real, generated.detach())\n            disc = disc + discriminator_loss(dr, df)[0]\n        if not torch.isfinite(disc):\n            raise RuntimeError(\'Non-finite discriminator loss\')\n        disc.backward()\n        grad_norm(mpd), grad_norm(msd)  # Check that both discriminators receive gradients.\n        dnorm, dnorm_after = clip_optimizer_gradients((mpd, msd), args.grad_clip)\n        od.step()\n        od.zero_grad(set_to_none=True)\n        for d in (mpd, msd):\n            set_requires_grad(d, False)\n            d.eval()  # Do not update spectral-norm buffers during generator pass.\n        og.zero_grad(set_to_none=True)\n        scale = gan_scale(step, args.gan_warmup_steps, args.gan_ramp_steps)\n        adv, fm = generated.new_zeros(()), generated.new_zeros(())\n        if scale > 0:\n            for d in (mpd, msd):\n                _, df, fr, ff = d(real, generated)\n                adv = adv + generator_loss(df)[0]\n                fm = fm + feature_loss(fr, ff) / 2  # vendor already multiplies by two\n            del fr, ff\n        mel = F.l1_loss(loss_mel(generated), loss_mel(real)).to(device)\n        with torch.no_grad():\n            target_content = content.encode_wave(real.squeeze(1))\n        generated_content = content.encode_wave(generated.squeeze(1))  # MUST keep autograd\n        perceptual = F.l1_loss(generated_content, target_content).to(device)\n        terms = dict(adversarial=scale*args.adv_weight*adv, feature_matching=scale*args.fm_weight*fm,\n                     mel=args.mel_weight*mel, content=args.content_weight*perceptual)\n        total = sum(terms.values())\n        if not torch.isfinite(total):\n            raise RuntimeError(\'Non-finite generator loss\')\n        for name, term in [(\'content\', perceptual)] + ([(\'adversarial\', adv)] if scale > 0 else []):\n            if name not in gradient_checks:\n                grad = torch.autograd.grad(term, predicted, retain_graph=True)[0]\n                value = float(grad.norm())\n                if not np.isfinite(value) or value <= 0:\n                    raise RuntimeError(f\'{name} has no finite gradient to mapper output\')\n                gradient_checks[name] = dict(step=step, to_mapper_output_grad_norm=value)\n                del grad\n                (out / \'gradient_checks.json\').write_text(json.dumps(gradient_checks, indent=2))\n                print(\'GRADIENT_CHECK \' + json.dumps(gradient_checks), flush=True)\n        loss_gradients = None\n        if (step in {start+1, args.gan_warmup_steps+1, args.gan_warmup_steps+args.gan_ramp_steps}\n                or (args.loss_grad_interval and step % args.loss_grad_interval == 0)):\n            loss_gradients = weighted_loss_gradients(terms, predicted)\n        total.backward()\n        mnorm, gnorm = grad_norm(mapper), grad_norm(generator)\n        if any(p.grad is not None or p.requires_grad for p in content.model.parameters()):\n            raise RuntimeError(\'Frozen wav2vec2 received parameter gradients\')\n        gnorm_combined, gnorm_after = clip_optimizer_gradients((mapper, generator), args.grad_clip)\n        mnorm_after, generator_norm_after = grad_norm(mapper), grad_norm(generator)\n        og.step()\n        synchronize(device)\n        elapsed = time.perf_counter() - started\n        times.append(elapsed)\n        record = dict(step=step, utterance_id=row[\'utterance_id\'], total=float(total.detach()), discriminator=float(disc.detach()),\n                      adversarial=float(adv.detach()), feature_matching_raw=float(fm.detach()), mel=float(mel.detach()), content=float(perceptual.detach()),\n                      mapper_grad=mnorm, generator_grad=gnorm, discriminator_grad=dnorm, seconds=elapsed,\n                      mapper_grad_post_clip=mnorm_after, generator_grad_post_clip=generator_norm_after,\n                      generator_optimizer_grad_pre_clip=gnorm_combined, generator_optimizer_grad_post_clip=gnorm_after,\n                      discriminator_grad_post_clip=dnorm_after, grad_clip=args.grad_clip,\n                      gan_scale=scale, adversarial_weight=scale*args.adv_weight, feature_matching_weight=scale*args.fm_weight,\n                      gan_losses_evaluated=scale > 0, weighted_losses={k: float(v.detach()) for k, v in terms.items()})\n        if loss_gradients is not None:\n            record[\'weighted_loss_grad_at_mapper_output\'] = loss_gradients\n        with (out / \'losses.jsonl\').open(\'a\') as f:\n            f.write(json.dumps(record) + \'\\n\')\n        if step == start+1 or step % args.log_interval == 0:\n            print(\'TRAIN \' + json.dumps(record), flush=True)\n        # Drop graphs before validation/checkpoint serialization.\n        del generated, real, generated_content, target_content, total, terms, term, adv, fm, mel, perceptual, disc, predicted, dr, df, inputs\n        og.zero_grad(set_to_none=True)\n        if device.type == "mps":\n            torch.mps.empty_cache()\n        if step == args.max_steps or (args.validation_interval and step % args.validation_interval == 0):\n            validations.append(validate(step, val_rows, get_item, mapper, generator, device, out, baseline))\n        if step == args.max_steps or (args.checkpoint_interval and step % args.checkpoint_interval == 0):\n            save_joint(out, step, mapper, generator, mpd, msd, og, od, args, config, init, rng)\n    summary = dict(steps=args.max_steps-start, mean_seconds=float(np.mean(times)),\n                   median_seconds=float(np.median(times)), all_losses_finite=True, output=str(out),\n                   validation_trend=[{k: v for k, v in r.items() if k != \'items\'} for r in validations])\n    (out / \'summary.json\').write_text(json.dumps(summary, indent=2))\n    print(\'FINISHED \' + json.dumps(summary), flush=True)\n\n\ndef parse_args(argv=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--device\', choices=[\'mps\',\'cpu\',\'cuda\',\'auto\'], default=\'mps\')\n    p.add_argument(\'--content-device\', choices=[\'cpu\', \'mps\', \'cuda\'], default=\'cpu\', help=\'CPU conserves unified GPU memory; waveform gradients still flow\')\n    p.add_argument(\'--manifest\', default=\'data/manifest_w5.csv\')\n    p.add_argument(\'--splits\', default=\'data/splits_w5.json\')\n    p.add_argument(\'--mapper-checkpoint\')\n    p.add_argument(\'--vocoder-init\', choices=[\'thai\',\'universal\'], default=\'thai\')\n    p.add_argument(\'--discriminator-checkpoint\')\n    p.add_argument(\'--output-dir\')\n    p.add_argument(\'--resume\', help=\'Joint checkpoint to resume into a NEW output directory\')\n    p.add_argument(\'--max-steps\', type=int, default=100)\n    p.add_argument(\'--segment-samples\', type=int, default=16384)\n    p.add_argument(\'--mapper-lr\', type=float, default=1e-5)\n    p.add_argument(\'--generator-lr\', type=float, default=1e-5)\n    p.add_argument(\'--discriminator-lr\', type=float, default=1e-5)\n    p.add_argument(\'--adv-weight\', type=float, default=1)\n    p.add_argument(\'--fm-weight\', type=float, default=2)\n    p.add_argument(\'--mel-weight\', type=float, default=45)\n    p.add_argument(\'--content-weight\', type=float, default=0.5, help=\'Conservative content anchor; inspect weighted-loss gradient diagnostics when tuning\')\n    p.add_argument(\'--grad-clip\', type=float, default=1.0, help=\'Combined max L2 norm per optimizer, before its step\')\n    p.add_argument(\'--gan-warmup-steps\', type=int, default=1000, help=\'Reconstruction-only generator updates; discriminators still train\')\n    p.add_argument(\'--gan-ramp-steps\', type=int, default=1000, help=\'Linear GAN/FM ramp after warmup (0 means immediate full weight)\')\n    p.add_argument(\'--loss-grad-interval\', type=int, default=1000, help=\'Log each weighted loss gradient at mapper output; 0 disables periodic checks\')\n    p.add_argument(\'--checkpoint-interval\', type=int, default=500)\n    p.add_argument(\'--validation-interval\', type=int, default=1000)\n    p.add_argument(\'--val-items\', type=int, default=3)\n    p.add_argument(\'--log-interval\', type=int, default=10)\n    p.add_argument(\'--seed\', type=int, default=1234)\n    return p.parse_args(argv)\n\n\nif __name__ == \'__main__\':\n    train(parse_args())\n'
EVALUATOR_SOURCE = '"""Real three-way full-pipeline TEST evaluation of an explicitly selected joint snapshot."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gc\nimport json\nfrom pathlib import Path\nimport sys\n\nimport librosa\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom tqdm.auto import tqdm\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom scripts.evaluate_vocoder_finetune import select_rows, METRICS, digest\nfrom src.evaluation.metrics import load_audio_16k\nfrom src.inference.run import (\n    AttrDict, Generator, Wav2Vec2ContentEncoder, load_mapper,\n    compute_mel_frame_count, interpolate_embedding, MEL_CLAMP_MIN, MEL_CLAMP_MAX,\n)\n\n\ndef evaluate(joint_checkpoint, output, device):\n    joint_checkpoint, output = Path(joint_checkpoint).resolve(), Path(output).resolve()\n    if not joint_checkpoint.is_file():\n        raise FileNotFoundError(joint_checkpoint)\n    # Never mix or overwrite the previous experiment\'s generated files.\n    output.mkdir(parents=True, exist_ok=False)\n    torch.set_num_threads(4)\n    mapper_path = ROOT / \'results/checkpoints/mapper_layer9.pt\'\n    variants = {\n        \'baseline\': ROOT / \'vendor/hifi-gan/checkpoints/UNIVERSAL_V1/g_02500000\',\n        \'approach_a\': ROOT / \'results/checkpoints/hifigan_thai/g_00010000\',\n        \'joint_finetuned\': joint_checkpoint,\n    }\n    configs = {name: json.loads(p.with_name(\'config.json\').read_text())\n               for name, p in variants.items() if name != \'joint_finetuned\'}\n    assert configs[\'baseline\'] == configs[\'approach_a\']\n    sample_rate = int(configs[\'baseline\'][\'sampling_rate\'])\n    rows = select_rows()\n    assert len(rows) == 8\n    provenance = dict(joint_checkpoint=str(joint_checkpoint), joint_sha256=digest(joint_checkpoint),\n                      original_mapper=str(mapper_path), original_mapper_sha256=digest(mapper_path),\n                      device=str(device), utterances=rows,\n                      checkpoints={k: dict(path=str(p), sha256=digest(p)) for k, p in variants.items()},\n                      protocol=\'Saved PCM_16 -> 16 kHz; shared-length truncation; no gain/time alignment\')\n    (output / \'provenance.json\').write_text(json.dumps(provenance, indent=2, ensure_ascii=False)+\'\\n\')\n    encoder = Wav2Vec2ContentEncoder(device=device, layer=9)\n    encoder.model.eval().requires_grad_(False)\n    references = {}\n    for row in rows:\n        folder = output / row[\'utterance_id\']\n        folder.mkdir()\n        clean, sr = sf.read(ROOT / row[\'clean_path\'], dtype=\'float32\', always_2d=True)\n        clean = clean.mean(axis=1)\n        if sr != sample_rate:\n            clean = librosa.resample(clean, orig_sr=sr, target_sr=sample_rate)\n        sf.write(folder/\'ground_truth.wav\', clean, sample_rate, subtype=\'PCM_16\')\n        references[row[\'utterance_id\']] = load_audio_16k(folder/\'ground_truth.wav\')\n    results = []\n    with (output/\'results.csv\').open(\'w\', newline=\'\') as handle, tqdm(total=24, unit=\'run\') as progress:\n        writer = csv.DictWriter(handle, fieldnames=[\'utterance_id\', \'variant\', \'stoi\', \'pesq\', \'snr\'])\n        writer.writeheader()\n        for variant, checkpoint in variants.items():\n            state = torch.load(checkpoint, map_location=\'cpu\', weights_only=True,\n                               mmap=(variant == \'joint_finetuned\'))\n            config = dict(configs[\'baseline\'])\n            mapper = load_mapper(mapper_path, device, ROOT/\'configs/model.yaml\', ROOT/\'configs/train.yaml\', layer=9)\n            if variant == \'joint_finetuned\':\n                mapper.load_state_dict(state[\'mapper\'], strict=True)\n                config = dict(state.get(\'vocoder_config\', config))\n                assert int(config[\'sampling_rate\']) == sample_rate\n                print(\'Joint checkpoint step:\', state[\'step\'], flush=True)\n            generator = Generator(AttrDict(config)).to(device)\n            generator.load_state_dict(state[\'generator\'], strict=True)\n            del state\n            mapper.eval().requires_grad_(False)\n            generator.eval().requires_grad_(False)\n            generator.remove_weight_norm()\n            for row in rows:\n                uid = row[\'utterance_id\']\n                progress.set_postfix(variant=variant, utterance=uid)\n                distorted = ROOT / row[\'distorted_path\']\n                with torch.inference_mode():\n                    embedding = encoder.encode(str(distorted))\n                    frames = compute_mel_frame_count(str(distorted))\n                    inputs = interpolate_embedding(embedding, frames).unsqueeze(0).to(device)\n                    mel = mapper(inputs).clamp(MEL_CLAMP_MIN, MEL_CLAMP_MAX).transpose(1, 2)\n                    audio = generator(mel).squeeze().cpu().numpy().astype(np.float32)\n                if audio.ndim != 1 or not audio.size or not np.isfinite(audio).all():\n                    raise RuntimeError(f\'Invalid generated waveform: {variant}/{uid}\')\n                wav = output / uid / f\'{variant}.wav\'\n                sf.write(wav, np.clip(audio, -1, 1), sample_rate, subtype=\'PCM_16\')\n                estimate = load_audio_16k(wav)\n                scores = {k: float(fn(references[uid], estimate)) for k, fn in METRICS.items()}\n                if not all(np.isfinite(v) for v in scores.values()):\n                    raise RuntimeError(f\'Non-finite metrics: {variant}/{uid}: {scores}\')\n                result = dict(utterance_id=uid, variant=variant, **scores)\n                results.append(result)\n                writer.writerow(result)\n                handle.flush()\n                progress.update(1)\n                del embedding, inputs, mel, audio\n            del mapper, generator\n            gc.collect()\n            if device.type == \'cuda\':\n                torch.cuda.empty_cache()\n    assert len(results) == 24\n    summary = {}\n    print(\'\\nVariant            STOI mean +/- std       PESQ mean +/- std       SNR dB mean +/- std\')\n    for variant in variants:\n        subset = [r for r in results if r[\'variant\'] == variant]\n        summary[variant] = {k: dict(mean=float(np.mean([r[k] for r in subset])),\n                                    std=float(np.std([r[k] for r in subset], ddof=1))) for k in METRICS}\n        print(f\'{variant:<18} \' + \'   \'.join(f\'{v["mean"]:.6f} +/- {v["std"]:.6f}\' for v in summary[variant].values()))\n    (output/\'summary.json\').write_text(json.dumps(summary, indent=2)+\'\\n\')\n    return summary\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--joint-checkpoint\', type=Path, required=True)\n    parser.add_argument(\'--output-dir\', type=Path, required=True, help=\'Must be a NEW directory\')\n    parser.add_argument(\'--device\', choices=[\'cpu\', \'cuda\', \'mps\'], default=\'cuda\')\n    args = parser.parse_args()\n    evaluate(args.joint_checkpoint, args.output_dir, torch.device(args.device))\n'

import hashlib, uuid
from datetime import datetime
trainer_hash = hashlib.sha256(TRAINER_SOURCE.encode()).hexdigest()
trainer_name = 'joint_finetune_stable_' + trainer_hash[:12]
trainer_path = REPO / 'src/training' / (trainer_name + '.py')
if trainer_path.exists():
    assert trainer_path.read_text() == TRAINER_SOURCE
else:
    trainer_path.write_text(TRAINER_SOURCE)
eval_hash = hashlib.sha256(EVALUATOR_SOURCE.encode()).hexdigest()
eval_path = REPO / 'scripts' / ('evaluate_joint_stable_' + eval_hash[:12] + '.py')
if eval_path.exists():
    assert eval_path.read_text() == EVALUATOR_SOURCE
else:
    eval_path.write_text(EVALUATOR_SOURCE)
print('Trainer source SHA-256:', trainer_hash)
subprocess.run([sys.executable, '-m', 'src.training.' + trainer_name, '--help'], check=True)

In [ ]:
# A NEW experiment from the original mapper + Approach-A vocoder; no resume.
SMOKE_OUTPUT = output_link / ('stable_smoke500_cuda_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8])
command = [sys.executable, '-u', '-m', 'src.training.' + trainer_name,
           '--device', 'cuda', '--content-device', 'cuda',
           '--mapper-checkpoint', 'results/checkpoints/mapper_layer9.pt',
           '--max-steps', '500', '--gan-warmup-steps', '100', '--gan-ramp-steps', '200',
           '--grad-clip', '1.0', '--content-weight', '0.5',
           '--validation-interval', '100', '--val-items', '3',
           '--loss-grad-interval', '100', '--checkpoint-interval', '250',
           '--log-interval', '10', '--output-dir', str(SMOKE_OUTPUT)]
print('Outputs directly on Drive:', SMOKE_OUTPUT.resolve())
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
# Actual observed gradient ranges and validation trend, not synthetic values.
import pandas as pd
losses = pd.read_json(SMOKE_OUTPUT / 'losses.jsonl', lines=True)
validation = pd.read_json(SMOKE_OUTPUT / 'validation.jsonl', lines=True)
assert len(losses) == 500 and int(losses.iloc[-1]['step']) == 500
assert losses.iloc[-1]['gan_scale'] == 1.0
fields = ['mapper_grad', 'generator_grad', 'generator_optimizer_grad_pre_clip',
          'mapper_grad_post_clip', 'generator_grad_post_clip',
          'generator_optimizer_grad_post_clip', 'discriminator_grad', 'discriminator_grad_post_clip']
print(losses[fields].agg(['min', 'median', 'max']).T.to_string())
print(validation[['step', 'stoi', 'pesq', 'delta_stoi', 'delta_pesq']].to_string(index=False))
print('Training steps/sec:', 1 / losses['seconds'].mean())
print('Raw gradient size alone is not a stability verdict; compare VAL trends to step 0.')
print('No longer run is launched automatically.')

In [ ]:
# Post-smoke TEST evaluation only; do not tune training against these eight IDs.
EVAL_OUTPUT = DRIVE_ROOT / 'eval_stable_joint' / SMOKE_OUTPUT.name
subprocess.run([sys.executable, '-u', str(eval_path), '--device', 'cuda',
                '--joint-checkpoint', str(SMOKE_OUTPUT / 'joint_00000500.pt'),
                '--output-dir', str(EVAL_OUTPUT)], cwd=REPO, check=True)
print('Real WAVs, results.csv and summary.json:', EVAL_OUTPUT)